In [ ]:
# Clone the rain repo
!git clone https://github.com/HemaBangaram/sim2real-rain-addon.git
import os
os.chdir('/kaggle/working/sim2real-rain-addon')
!ls  # should show: training  testing


In [ ]:
# Install same libraries as offroad
!pip install -q diffusers==0.27.2 transformers accelerate xformers \
             huggingface_hub bitsandbytes Pillow torchvision tqdm


In [ ]:
# Install evaluation dependencies
!pip install -q lpips torch-fidelity
print('Done')


In [ ]:
import os, json
os.chdir('/kaggle/working/sim2real-rain-addon')

# Reset to clean version
!git checkout training/training_rain_sim.ipynb

notebook_path = 'training/training_rain_sim.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

for cell in nb_data['cells']:
    if cell['cell_type'] == 'code':
        source_text = ''.join(cell['source'])

        # Patch 1: CLIPFeatureExtractor (same fix as offroad)
        if 'CLIPFeatureExtractor' in source_text and 'CLIPImageProcessor as' not in source_text:
            source_text = source_text.replace(
                'CLIPFeatureExtractor',
                'CLIPImageProcessor as CLIPFeatureExtractor'
            )

        # Patch 2: Disable intermediate saving (same fix as offroad)
        if 'pipeline.save_pretrained(save_path)' in source_text and 'pass #' not in source_text:
            source_text = source_text.replace(
                'pipeline.save_pretrained(save_path)',
                'pass # pipeline.save_pretrained(save_path) - Disabled to save disk space!'
            )

        cell['source'] = [source_text]

with open(notebook_path, 'w', encoding='utf-8') as f:
    json.dump(nb_data, f, indent=1)

print('Notebook patched successfully!')


In [ ]:
# Run training (same command as offroad)
!jupyter nbconvert --to notebook --execute \
    training/training_rain_sim.ipynb \
    --output training_rain_sim_done.ipynb \
    --ExecutePreprocessor.timeout=3600

print('Training Complete!')


In [ ]:
# Run inference (same command as offroad)
os.chdir('/kaggle/working/sim2real-rain-addon/testing')
!python testing_rain_real2sim.py
print('Inference Complete!')


In [ ]:
# Show all output images - i1 / i2 / i3 per row
import matplotlib.pyplot as plt
from PIL import Image
import glob

input_folder  = 'input_rain_real'
style_folder  = 'style_rain_sim'
output_folder = 'output_rain_real2sim'

input_files  = sorted([f for f in os.listdir(input_folder)  if f.lower().endswith(('.png','.jpg','.jpeg'))])
style_files  = sorted([f for f in os.listdir(style_folder)  if f.lower().endswith(('.png','.jpg','.jpeg'))])
output_files = sorted([f for f in os.listdir(output_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))])

n = len(output_files)
fig, axes = plt.subplots(n, 3, figsize=(18, 6 * n))
if n == 1: axes = [axes]

for i, fname in enumerate(output_files):
    axes[i][0].imshow(Image.open(os.path.join(input_folder,  input_files[i])))
    axes[i][0].set_title(f'i1: Real Rain Input\n{fname}', fontsize=9); axes[i][0].axis('off')
    axes[i][1].imshow(Image.open(os.path.join(style_folder,  style_files[0])))
    axes[i][1].set_title(f'i2: Rain Sim Style Ref', fontsize=9); axes[i][1].axis('off')
    axes[i][2].imshow(Image.open(os.path.join(output_folder, fname)))
    axes[i][2].set_title(f'i3: Output (Sim Style)\n{fname}', fontsize=9); axes[i][2].axis('off')

plt.suptitle('Rain Real2Sim — All Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_folder, '_all_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Showed {n} images')


In [ ]:
# Run evaluation (same command as offroad)
os.chdir('/kaggle/working/sim2real-rain-addon')

!jupyter nbconvert --to notebook --execute \
    testing/evaluation.ipynb \
    --output testing/evaluation_done.ipynb \
    --ExecutePreprocessor.timeout=3600

print('Evaluation Complete!')


In [ ]:
# Zip results (same as offroad)
os.chdir('/kaggle/working/')

!zip -r rain_real2sim_results.zip \
    sim2real-rain-addon/training/training_rain_sim_done.ipynb \
    sim2real-rain-addon/testing/evaluation_done.ipynb \
    sim2real-rain-addon/testing/output_rain_real2sim/

print('Zip created!')
from IPython.display import FileLink
display(FileLink('rain_real2sim_results.zip'))
